### Note: Please ensure that this notebook is run in the daq environment

In [1]:
import glob
import os
import tqdm
from collections import defaultdict
import re
import numpy as np
import pandas as pd
from scipy.io import wavfile
# from moviepy.video.io.ffmpeg_tools import ffmpeg_extract_subclip
from moviepy.video.io.VideoFileClip import VideoFileClip
from datetime import datetime, timedelta

In [2]:
def seconds_to_hms(seconds):
    # Calculate hours, minutes, and seconds
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    # Format the time in HH:MM:SS
    return f"{int(hours):02}:{int(minutes):02}:{int(seconds):02}.{int(remainder%1*1000):03}"

In [3]:
from moviepy.tools import subprocess_call
from moviepy.config import get_setting

# def ffmpeg_extract_subclip(filename, t1, t2, targetname=None):
def ffmpeg_extract_subclip(filename, start_frame, end_frame, targetname=None):
    """ Makes a new video file playing video file ``filename`` between
    the times ``t1`` and ``t2``. """
    # !ffmpeg -i {filename} -ss {seconds_to_hms(t1)} -to {seconds_to_hms(t2)} -c copy {targetname}
    tmp_str = f"between(n\,{start_frame}\,{end_frame}),setpts=PTS-STARTPTS"
    !ffmpeg -i {filename} -vf select={tmp_str} -c:v libx264 -crf 0 {targetname}


In [4]:
experiment_no = 547
no_channels = 14
camera_fps = 30
# camera_corrected_sample_time = 0.03338542
file_length = 360 # in seconds (360 sec - 6 mins)
# nidaq_folder = "D:/big_setup/experiment_{}/nidaq/".format(experiment_no)
nidaq_folder = "D:/big_setup/experiment_{}/nidaq/".format(experiment_no)
video_folder = "D:/big_setup/experiment_{}/videos/".format(experiment_no)

In [5]:
# Deleting the trunc files if it is already present
trunc_files = glob.glob(nidaq_folder+"*_trunc*")
for i in trunc_files:
    os.remove(i) 

trunc_files = glob.glob(video_folder+"*_trunc*")
for i in trunc_files:
    os.remove(i) 

In [6]:
nidaq_files = glob.glob(nidaq_folder+"*")
video_files = glob.glob(video_folder+"*")

In [7]:
# Creating separate folders for the concatenated data
path_concat = "D:/big_setup/experiment_{}/concatenated_data_cam_mic_sync/temp/".format(experiment_no)

try:
    os.makedirs(path_concat)
except:
    pass

In [ ]:
path_final = "D:/big_setup/experiment_{}/concatenated_data_cam_mic_sync/".format(experiment_no)
# Functions to sort file names correctly
def atoi(text):
    return int(text) if text.isdigit() else text

def natural_keys(text):
    return [ atoi(c) for c in re.split(r'(\d+)', text) ]
nidaq_files.sort(key=natural_keys)
video_files.sort(key=natural_keys)
nidaq_files = [i.replace("\\",'/') for i in nidaq_files]
video_files = [i.replace("\\",'/') for i in video_files]
### For Videos
# Getting the camera names
temp_cam_name = []
for i in video_files:
    temp_cam_name.append(i.split("/")[-1].split("-")[0])
camera_names = list(set(temp_cam_name))
print(camera_names)
# Creating a dictionary of all videos of a single camera
video_by_camera = defaultdict(lambda:[])
for idx,name in enumerate(temp_cam_name):
    video_by_camera[name].append(video_files[idx])
cam_clk_data = pd.read_csv(f"D:/big_setup/experiment_{experiment_no}/camera_timestamps.csv")
file_breaks  = np.arange(0,cam_clk_data.tail(1).index[0]/camera_fps,file_length)
timestamp_idx = []
for val in file_breaks:
    timestamp_idx.append((np.abs(cam_clk_data[f'concat_{camera_names[0]}_time_from_vid_start'] - val)).argmin())
for cam in camera_names:
    file_break_idx = 1
    file_concat_index = 0 
    str_1 = "%03d"%(file_concat_index)
    filename_txt = path_concat + f"video_{cam}_{str_1}.txt"
    f = open(filename_txt, 'w')
    no_videos = len(video_by_camera[cam])

    for index, vid_name in enumerate(video_by_camera[cam]):


        try:
            temp = cam_clk_data.iloc[timestamp_idx[file_break_idx]][f'{cam}_file_name'].split('\\')
        except:
            print("Index from timestamp not found")
            pass
        
        if temp[0]+'/'+temp[1] == vid_name and index != no_videos -1 :
            #break_time = cam_clk_data.iloc[timestamp_idx[file_break_idx]][f'{cam}_time_from_vid_start']
            break_frame = cam_clk_data.iloc[timestamp_idx[file_break_idx]][f'{cam}_frame_idx']
            target_file = temp[0]+"/"+temp[1].split(".")[0]+"_trunc_0.mp4"
            ffmpeg_extract_subclip(vid_name, 0, break_frame, targetname=target_file)
            
            # with VideoFileClip(vid_name) as video:
            #     new = video.subclip(0.0, break_time)
            #     new.write_videofile(target_file, audio_codec='aac')

            f.write("file \'{}\'\n".format(target_file))
            f.close()


            target_file = temp[0]+"/"+temp[1].split(".")[0]+"_trunc_1.mp4"
            with VideoFileClip(vid_name) as video:
                #end_time = video.duration
                end_frame = video.reader.nframes
                # new = video.subclip(break_time, end_time)
                # new.write_videofile(target_file, audio_codec='aac')
            ffmpeg_extract_subclip(vid_name, break_frame, end_frame, targetname=target_file)



            file_concat_index+=1
            file_break_idx+=1
            str_1 = "%03d"%(file_concat_index)
            filename_txt = path_concat + f"video_{cam}_{str_1}.txt"
            f = open(filename_txt, 'w')
            f.write("file \'{}\'\n".format(target_file))


        elif temp[0]+'/'+temp[1] != vid_name and index == no_videos -1:
            cam_stop_rec_video_name = cam_clk_data[f"{cam}_file_name"].iloc[-1]
            try:
                cam_stop_rec_frame = int(cam_clk_data[f"{cam}_frame_idx"].iloc[-1])
            except Exception as e: # trying to catch exceptions that arise when the video is not long enough (camera drop/frame drop)
                print(e)
                cam_stop_rec_frame = None
            
            target_file = vid_name.split(".")[0]+"_trunc_0.mp4"

            if cam_stop_rec_frame == None:
                target_file = cam_stop_rec_video_name
            else:
                ffmpeg_extract_subclip(cam_stop_rec_video_name, 0, cam_stop_rec_frame, targetname=target_file)
                
                # with VideoFileClip(cam_stop_rec_video_name) as video:
                #     new = video.subclip(0.0,(cam_stop_rec_frame+1)/camera_fps)
                #     new.write_videofile(target_file, audio_codec='aac')

            f.write("file \'{}\'\n".format(target_file))
            f.close()

        elif temp[0]+'/'+temp[1] == vid_name and index == no_videos -1:
            cam_stop_rec_video_name = cam_clk_data[f"{cam}_file_name"].iloc[-1]
            # cam_stop_rec_frame = int(cam_clk_data[f"{cam}_frame_idx"].iloc[-1])

            try:
                cam_stop_rec_frame = int(cam_clk_data[f"{cam}_frame_idx"].iloc[-1])
            except Exception as e: # trying to catch exceptions that arise when the video is not long enough (camera drop/frame drop)
                print(e)
                cam_stop_rec_frame = None

            #break_time = cam_clk_data.iloc[timestamp_idx[file_break_idx]][f'{cam}_time_from_vid_start']
            break_frame = cam_clk_data.iloc[timestamp_idx[file_break_idx]][f'{cam}_frame_idx']
            target_file = temp[0]+"/"+temp[1].split(".")[0]+"_trunc_0.mp4"
            if cam_stop_rec_frame == None:
                target_file = cam_stop_rec_video_name
            else:
                ffmpeg_extract_subclip(vid_name, 0, break_frame, targetname=target_file)
                # with VideoFileClip(vid_name) as video:
                #     new = video.subclip(0.0,break_time)
                #     new.write_videofile(target_file, audio_codec='aac')

            f.write("file \'{}\'\n".format(target_file))
            f.close()

            if cam_stop_rec_frame != None:
                target_file = temp[0]+"/"+temp[1].split(".")[0]+"_trunc_1.mp4"
                #end_time = (cam_stop_rec_frame+1)/camera_fps
                end_frame = cam_stop_rec_frame
                ffmpeg_extract_subclip(vid_name, break_frame, end_frame, targetname=target_file)
                # with VideoFileClip(vid_name) as video:
                #     new = video.subclip(break_time,end_time)
                #     new.write_videofile(target_file, audio_codec='aac')
                
                file_concat_index+=1
                str_1 = "%03d"%(file_concat_index)
                filename_txt = path_concat + f"video_{cam}_{str_1}.txt"
                f = open(filename_txt, 'w')
                f.write("file \'{}\'\n".format(target_file))
                f.close()

        elif index != no_videos -1:
            f.write("file \'{}\'\n".format(vid_name))

        
# # Reading the timestamps data for the camera clock channel 
# cam_clk_data = pd.read_csv(f"D:/big_setup/experiment_{experiment_no}/camera_timestamps.csv")
# for cam in camera_names:
#     no_videos = len(video_by_camera[cam])
#     filename_txt = path_concat + "video_{}.txt".format(cam)
#     f = open(filename_txt, 'w')
#     for index, vid_name in enumerate(video_by_camera[cam]):
#         if index != no_videos -1 : 
#             f.write("file \'{}\'\n".format(vid_name))
#         else: # truncating the last few seconds of the last video clip based on the timestamps generated
#             cam_stop_rec_video_name = cam_clk_data[f"{cam}_file_name"].iloc[-1]
#             cam_stop_rec_frame = int(cam_clk_data[f"{cam}_frame_idx"].iloc[-1])
#             print(cam_stop_rec_frame)
#             print(cam_stop_rec_video_name)
#             ffmpeg_extract_subclip(cam_stop_rec_video_name, 0.0, (cam_stop_rec_frame+1)/camera_fps, targetname=cam_stop_rec_video_name.split("\\")[0]+"/"+cam_stop_rec_video_name.split("\\")[1].split(".")[0]+"_trunc.mp4")
#             f.write("file \'{}\'\n".format(cam_stop_rec_video_name.split("\\")[0]+"/"+cam_stop_rec_video_name.split("\\")[1].split(".")[0]+"_trunc.mp4"))
 
#     f.close()

# Used for testing
# cam_stop_rec_video_name.split("\\")[0]+"/"+cam_stop_rec_video_name.split("\\")[1].split(".")[0]+"_trunc.mp4"
# for item in tqdm.tqdm(video_by_camera.items()):
#     fps = 30
#     resolution = (1600,1200) 
    
#     filename_txt = path_concat + "video_{}.txt".format(item[0])
#     f = open(filename_txt, 'w')

#     for idx, vid in enumerate(item[1]):
#             # if idx != (len(item[1])-1):
#             f.write("file \'{}\'\n".format(vid))
#     f.close()
    
### For NIDAQ
# Reading the timestamps data for the camera clock channel 
audio_clk_data = pd.read_csv(f"D:/big_setup/experiment_{experiment_no}/camera_timestamps.csv")
audio_start_rec_idx = int(cam_clk_data["clk_ch_file_name"][0].split("\\")[1].split("_")[2])
audio_stop_rec_idx = int(cam_clk_data["clk_ch_file_name"].iloc[-1].split("\\")[1].split("_")[2])
start_sample_index =  int(cam_clk_data["clk_ch_sample_idx"][0])
end_sample_index =  int(cam_clk_data["clk_ch_sample_idx"].iloc[-1])
# Concatenating the microphone channels
for i in range(no_channels):
    file_break_idx = 1
    file_concat_index = 0 # index for the text files used to concatenate data
    flag = 0 # to indicate camera start record index has reached
    str_1 = "%02d"%(i)
    str_2 = "%03d"%(file_concat_index)
    filename_txt = path_concat + f"channel_{str_1}_file_{str_2}.txt"
    f = open(filename_txt, 'w')
    for idx,j in tqdm.tqdm(enumerate(nidaq_files[i::no_channels])):
            
            try:
                break_channel_idx = int(audio_clk_data.iloc[timestamp_idx[file_break_idx]][f'clk_ch_file_name'].split('_')[-2])
            except:
                 pass

            if idx == audio_start_rec_idx:            
                # Loading the file
                sampling_rate,ch = wavfile.read(j)
                ch = ch[start_sample_index:]
                target_file = j.split(".")[0]+"_trunc_0.wav"
                wavfile.write(target_file,sampling_rate,ch)
                flag = 1
                f.write("file \'{}\'\n".format(target_file))

            elif idx == break_channel_idx and idx!= audio_stop_rec_idx:

                break_sample_idx = audio_clk_data.iloc[timestamp_idx[file_break_idx]][f'clk_ch_sample_idx']
    
                # Loading the file
                sampling_rate,ch = wavfile.read(j)
                ch = ch[:break_sample_idx]
                target_file = j.split(".")[0]+"_trunc_0.wav"
                wavfile.write(target_file,sampling_rate,ch)

                f.write("file \'{}\'\n".format(target_file))
                f.close()

                
                target_file = j.split(".")[0]+"_trunc_1.wav"
                # Loading the file
                sampling_rate,ch = wavfile.read(j)
                ch = ch[break_sample_idx:]
                wavfile.write(target_file,sampling_rate,ch)

                file_concat_index+=1
                file_break_idx+=1
                str_1 = "%02d"%(i)
                str_2 = "%03d"%(file_concat_index)
                filename_txt = path_concat + f"channel_{str_1}_file_{str_2}.txt"
                f = open(filename_txt, 'w')
                f.write("file \'{}\'\n".format(target_file))
     
            elif idx != break_channel_idx and idx == audio_stop_rec_idx:
                    
            
                target_file = j.split(".")[0]+"_trunc_0.wav"

                # Loading the file
                sampling_rate,ch = wavfile.read(j)
                ch = ch[:end_sample_index]
                wavfile.write(target_file,sampling_rate,ch)
                f.write("file \'{}\'\n".format(target_file))
                f.close()
                break

            elif idx == break_channel_idx and idx == audio_stop_rec_idx:

                break_sample_idx = audio_clk_data.iloc[timestamp_idx[file_break_idx]][f'clk_ch_sample_idx']


                target_file = j.split(".")[0]+"_trunc_0.wav"
                # Loading the file
                sampling_rate,ch = wavfile.read(j)
                ch = ch[:break_sample_idx]
                wavfile.write(target_file,sampling_rate,ch)
                f.write("file \'{}\'\n".format(target_file))
                f.close()

            
                target_file = j.split(".")[0]+"_trunc_1.wav"
                # Loading the file
                sampling_rate,ch = wavfile.read(j)
                ch = ch[break_sample_idx:end_sample_index]
                wavfile.write(target_file,sampling_rate,ch)

                file_concat_index+=1
                str_1 = "%02d"%(i)
                str_2 = "%03d"%(file_concat_index)
                filename_txt = path_concat + f"channel_{str_1}_file_{str_2}.txt"
                f = open(filename_txt, 'w')
                f.write("file \'{}\'\n".format(target_file))
                f.close()
                break

            elif flag == 1:
                f.write("file \'{}\'\n".format(j))
# # Concatenating the microphone channels
# flag = 0 # to indicate camera start record index has reached
# for i in range(no_channels):
#     concat_mic_channel = np.array([])
#     sampling_rate = 192000
#     filename_wav = path_concat + "channel_{}.txt".format(i)
#     f = open(filename_wav, 'w')
    
#     for idx,j in tqdm.tqdm(enumerate(nidaq_files[i::no_channels])):
#         if idx == cam_start_rec_idx:
#             # Loading the file
#             sampling_rate,ch = wavfile.read(j)
#             ch = ch[start_sample_index:]
#             wavfile.write(j.split(".")[0]+"_trunc.wav",sampling_rate,ch)
#             j = j.split(".")[0]+"_trunc.wav"
#             # file_lengths+=(300-)
#             flag = 1 
#         elif idx == cam_stop_rec_idx:
#             # Loading the file
#             sampling_rate,ch = wavfile.read(j)
#             ch = ch[:end_sample_index]
#             wavfile.write(j.split(".")[0]+"_trunc.wav",sampling_rate,ch)
#             j = j.split(".")[0]+"_trunc.wav"
#             flag = 2
#         if flag == 0:
#             continue
#         f.write("file \'{}\'\n".format(j))
#         if flag == 2:
#             break
    
#     f.close()

## Steps done via code in the below cell 
- open anaconda prompt
- Enter the following commands:
    - For video concatenation:
        - ```conda activate daq```
        - ```ffmpeg -f concat -safe 0 -i {path_to_text_file} -c copy {path_to_output_file}```
            
            eg: ```ffmpeg -f concat -safe 0 -i D:\big_setup\experiment_9\concatenated_data_cam_mic_sync\video_e3v83b3.txt -c copy D:\big_setup\experiment_9\concatenated_data_cam_mic_sync\e3v83b3_ffmpeg.mp4```
    - For NIDAQ concatenation:
        - - ```conda activate daq```
        - ```ffmpeg -f concat -safe 0 -i {path_to_text_file} -c copy {path_to_output_file}```
            
            eg: ```ffmpeg -f concat -safe 0 -i D:\big_setup\experiment_9\concatenated_data_cam_mic_sync\channel_0.txt -c copy D:\big_setup\experiment_9\concatenated_data_cam_mic_sync\channel_0_ffmpeg.wav```
no_audio_files = int(len(glob.glob(path_concat + "*channel_*.txt"))/no_channels)
no_video_files = int(len(glob.glob(path_concat + "*video_*.txt"))/len(camera_names))
timestamp_record_data = cam_clk_data.iloc[timestamp_idx]
time_starts = []
time_ends = []
column_names = [i for i in timestamp_record_data.columns if i.endswith("file_name") and not(i.startswith('clk'))]

starting_timestamp = datetime.strptime(timestamp_record_data[column_names[0]][0].split('\\')[-1].split('-')[1], '%Y%m%dT%H%M%S')

# for camera_file in column_names:
#     for file_name in timestamp_record_data[camera_file]:
#         time_starts.append(datetime.strptime(file_name.split('\\')[-1].split('-')[1], '%Y%m%dT%H%M%S'))
#         time_ends.append(time_starts[-1] + timedelta(minutes=file_length/60))
# if no_audio_files != no_video_files:
#     raise Exception("Number of audio and video files don't match")
# else:
matched_csv = []
timestamp_write = starting_timestamp
f = open(path_final+"README.txt", 'w')
f.write("The files that are synced are as follows:\n")
f.write("Video file names - corresponding audio file names - timestamp range of file \n")
for i in range(no_audio_files):
    temp_0 = []
    temp_1 = []
    temp_2 = [timestamp_write.strftime("%Y-%m-%d %H:%M:%S"),(timestamp_write + timedelta(minutes=file_length/60)).strftime("%Y-%m-%d %H:%M:%S")]
    timestamp_write += timedelta(minutes=file_length/60)
    for j in range(no_channels):
        var_1 = "%03d"%(i)
        var_2 = "%02d"%(j)
        temp_0.append(f"channel_{var_2}_file_{var_1}")
    for j in camera_names:
        var_1 = "%03d"%(i)
        temp_1.append(f"video_{j}_{var_1}")
    f.write(f"[{', '.join(temp_1)}] - [{', '.join(temp_0)}] - [{', '.join(temp_2)}]\n")
    matched_csv.append([temp_1,temp_0,temp_2])
pd.DataFrame(matched_csv,columns=["video","audio","timestamp"]).to_csv(path_final+"sync.csv", index = False)
f.close()
filename_txt = path_concat + "*.txt"
txt_files = glob.glob(filename_txt)
video_files = []
audio_files = []

for file in txt_files:
    name = file.split('\\')[-1].split(".")[0]
    full_path_txt = path_concat+name+".txt"
    full_path_txt = full_path_txt.replace("/","\\")
    if name[0] == "c":
        full_path_wav = path_final+name+".wav"
        full_path_wav = full_path_wav.replace("/","\\")
        !ffmpeg -f concat -safe 0 -i {full_path_txt} -c copy  {full_path_wav}
        # -rf64 auto
    else:
        full_path_mp4 = path_final+name+".mp4"
        full_path_mp4 = full_path_mp4.replace("/","\\")
        !ffmpeg -f concat -safe 0 -i {full_path_txt} -c copy {full_path_mp4}

## Testing sync

In [ ]:
'''import warnings
import math'''

In [ ]:
'''# Redoing the files that didn't get written
channel_txt_search = path_concat + "*channel_*.txt"
video_txt_search = path_concat + "*video_*.txt"

channel_txt_filenames = glob.glob(channel_txt_search)
video_txt_filenames = glob.glob(video_txt_search)

channel_txt_filenames.sort(key=natural_keys)
video_txt_filenames.sort(key=natural_keys)

upper_no_audio_files = math.ceil(len(channel_txt_filenames)/no_channels)
upper_no_video_files = math.ceil(len(video_txt_filenames)/len(camera_names))
print(f"Number of audio files: {upper_no_audio_files}")
print(f"Number of video files: {upper_no_video_files}")



# For channel txt files
prev_ch_idx_nos = 0
prev_ch_nos = 0

# if not(len(channel_txt_filenames)%no_channels > (no_channels-1)):
#     raise Exception(f"Channel txt files are missing re-run code")


# First test - no of indices per channel is the same
if len(channel_txt_filenames)%no_channels > 0:
    warnings.warn("The length of the audio channel text files do not match")
    print("Testing each channels txt files")

    counter = 0

    for name in channel_txt_filenames:
        vals = name.split("\\")[-1].split('_')
        channel_ = int(vals[1])
        index_ = int(vals[2].split(".")[0])

        if channel_ - prev_ch_nos == 1:
                prev_ch_nos = channel_
                prev_ch_idx_nos = 0
                counter = 0
        else:
            raise Exception(f"Missing channel {prev_ch_nos+1} - need to rerun code")

        if channel_ - prev_ch_nos == 0:
            if counter == 0:
                counter = 1
                if  index_ != prev_ch_idx_nos:
                    raise Exception(f"Missing channel {prev_ch_nos} index {prev_ch_idx_nos} - need to rerun code")
                
            if prev_ch_idx_nos - index_ > 1:
                    no_missing_files =  prev_ch_idx_nos - index_
                    raise Exception(f"There are {no_missing_files} text files in channel {channel_} between {prev_ch_idx_nos} and {index_} - need to rerun code") 

            
            prev_ch_idx_nos+=1

        
        temp = file.split('\\')[-1].split(".")[0]
        full_path_txt = path_concat+temp+".txt"
        full_path_txt = full_path_txt.replace("/","\\")
        full_path_wav = path_final+name+".wav"
        full_path_wav = full_path_wav.replace("/","\\")
        if  os.path.exists(full_path_wav):
            continue
        else:
            warnings.warn(f"File {temp}.wav doesn't exist - trying to rerun")
            !ffmpeg -f concat -safe 0 -i {full_path_txt} -c copy  {full_path_wav}
            if not(os.path.exists(full_path_wav)):
                   raise Exception("Unable to create file from txt, re run concat")
        
    

# For video txt files 
prev_vid_idx_nos = 0
camera_names.sort(key=natural_keys)
prev_vid_name = camera_names[0]


# if not(len(video_txt_filenames)/len(camera_names) > len(camera_names)-1):
#     raise Exception(f"Video txt files are missing re-run code")

# First test - no of indices per video is the same
if len(video_txt_filenames)%len(camera_names) > 0:
    warnings.warn("The length of the video text files per camera do not match")
    print("Testing each cameras txt files")

    counter = 0

    for name in video_txt_filenames:

        vals = name.split("\\")[-1].split('_')
        video_ = vals[1]
        index_ = int(vals[2].split(".")[0])

        if video_ != prev_vid_name:
                prev_vid_name = video_
                prev_vid_idx_nos = 0
                counter = 0

        if video_ == prev_vid_name:
            if counter == 0:
                counter = 1
                if  index_ != prev_vid_idx_nos:
                    raise Exception(f"Missing video {prev_vid_name} index {prev_vid_idx_nos}")
                
            if prev_vid_idx_nos - index_ > 1:
                    no_missing_files =  prev_vid_idx_nos - index_
                    raise Exception(f"There are {no_missing_files} text files in channel {video_} between {prev_vid_idx_nos} and {index_}") 

            
            prev_ch_idx_nos+=1


        temp = file.split('\\')[-1].split(".")[0]
        full_path_txt = path_concat+temp+".txt"
        full_path_txt = full_path_txt.replace("/","\\")

        full_path_mp4 = path_final+name+".mp4"
        full_path_mp4 = full_path_mp4.replace("/","\\")

        if  os.path.exists(full_path_mp4):
            continue
        else:
            warnings.warn(f"File {temp}.mp4 doesn't exist - trying to rerun")
            !ffmpeg -f concat -safe 0 -i {full_path_txt} -c copy  {full_path_mp4}
            if not(os.path.exists(full_path_wav)):
                   raise Exception("Unable to create file from txt, re run concat")'''

SyntaxError: invalid syntax (256407044.py, line 65)

old camera frame rate - 1/0.03338542

new camera frame rate - 